# Multi-Dimensional UMAP Export for Web Viewer

This notebook doubles as a tutorial for exporting deterministic 1D/2D/3D embeddings into the Cellucid WebGL viewer bundles.

> This variant is pre-configured for the **Human Developmental Cell Atlas (HDCA)** dataset; all source files live on the compute cluster at `/lustre/groups/ml01/workspace/kemal.inecik/hdca/`. Tweak the configuration cell if your files live elsewhere.

**In this walkthrough you will**
- configure dataset roots once so the same notebook runs without editing paths elsewhere.
- verify that expression counts (`X`) and the latent representation (`obsm['latent']`) are present before running anything expensive.
- remap `var.index` from Ensembl gene IDs to HGNC gene symbols so the viewer displays human-readable gene names.
- recompute missing UMAP dimensions only when necessary while reusing a single neighbor graph for perfect alignment across 1D/2D/3D.
- hydrate metadata + expressions and feed everything into `cellucid.prepare`.
- validate the generated binary assets and manifests before handing them to the frontend.

Each section calls out why the code exists so you can adapt the pattern to your own datasets.


## Environment

These setup helpers make the notebook location-agnostic: run it from the repo root, from `notebooks/`, or from VS Code and the imports/paths will still resolve.


In [ ]:
1

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import gc
import sys
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

HERE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(HERE)
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

sc.settings.verbosity = 3

## Configuration

Keep all project-specific paths and knobs together so rerunning exports becomes a one-cell edit exercise.

**Key differences from smaller datasets:**
- `SOURCE_FILE` lives on lustre (the server) rather than in the project `data/` tree — it is too large to copy locally.
- `EXPERIMENT_FILE` (the intermediate h5ad that stores computed UMAP embeddings) is written alongside the source file on lustre to avoid re-computation across sessions.
- Gene names: `var.index` in the source file contains Ensembl IDs; we remap to HGNC symbols via `var['hgnc']` before export.


In [ ]:
# Dataset slug — used to name the export directory
DATASET_NAME = "hdca"

# Source file: integration h5ad produced by the scVI/scANVI unification pipeline.
# Contains expression counts in X and the latent representation in obsm['latent'].
SOURCE_FILE = Path(
    "/lustre/groups/ml01/workspace/kemal.inecik/hdca/temp/models/"
    "20240613_152640100145_gpusrv44.scidom.de_2367215_unification_union_20241216_hvg-intersection_integration.h5ad"
)

# Intermediate file: written once UMAP embeddings have been computed.
# Saved next to the source file so we can skip recomputation on subsequent runs.
EXPERIMENT_FILE = SOURCE_FILE.parent / (
    SOURCE_FILE.stem + "_umap.h5ad"
)

# Export destination consumed by the Cellucid WebGL viewer
EXPORT_DIR = PROJECT_ROOT.parent / "cellucid-datasets" / "exports" / DATASET_NAME

print(f"SOURCE_FILE  : {SOURCE_FILE}")
print(f"EXPERIMENT_FILE: {EXPERIMENT_FILE}")
print(f"EXPORT_DIR   : {EXPORT_DIR}")

## Verify source data

Open the file in backed (memory-mapped) mode to confirm that:
- `X` (expression counts) is present and the right shape, and
- `obsm['latent']` (the scVI latent space) is present.

Both are required downstream — counts for the quantized gene-expression export, and the latent space for computing neighbors and centroid statistics.


In [ ]:
if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE_FILE}")

adata_preview = ad.read_h5ad(SOURCE_FILE, backed="r")
print(adata_preview)
print()

# --- Expression counts ---
if adata_preview.X is None:
    raise ValueError("X is None — expression counts are missing from the source file.")
print(f"✓ X present: shape {adata_preview.X.shape}, dtype {adata_preview.X.dtype}")

# --- Latent space ---
LATENT_KEY = "latent"
if LATENT_KEY not in adata_preview.obsm:
    raise KeyError(
        f"obsm['{LATENT_KEY}'] not found. Available keys: {list(adata_preview.obsm.keys())}"
    )
print(f"✓ obsm['{LATENT_KEY}'] present: shape {adata_preview.obsm[LATENT_KEY].shape}")

# --- var index (gene IDs before remapping) ---
print(f"\nvar.index (Ensembl IDs, first 5): {adata_preview.var.index[:6].tolist()}")
print(f"var['hgnc'] (gene symbols, first 5): {adata_preview.var['hgnc'][:6].tolist()}")
print(np.all(adata_preview.var.index==adata_preview.var['hgnc']))

# --- Observation metadata ---
print(f"\nobs columns: {adata_preview.obs.columns.tolist()}")
if "LVL0" in adata_preview.obs:
    print("\nLVL0 value counts:")
    for label, count in adata_preview.obs["LVL0"].value_counts().items():
        print(f"  {label}: {count:,}")

adata_preview.file.close()
del adata_preview

## UMAP parameters

All tunable knobs live here.  Change them only when you explicitly want a new layout — keeping them stable ensures reproducible exports between releases.


In [ ]:
# kNN graph parameters (shared across all UMAP dimensionalities)
n_neighbors = 15
min_dist = 0.5
RANDOM_SEED = 0

# Storage keys for each dimensionality we care about.
UMAP_DIMENSION_KEYS = {
    1: "X_umap_1d",
    2: "X_umap_2d",
    3: "X_umap_3d",
}


def compute_umap_embedding(adata_source, n_components: int, min_dist: float, random_state: int) -> np.ndarray:
    """Compute a UMAP embedding with the provided dimensionality without mutating the source AnnData."""
    neighbors_params = adata_source.uns.get("neighbors", {}).get("params", {})
    use_rep = neighbors_params.get("use_rep", None)

    adata_temp = ad.AnnData(
        obs=adata_source.obs[[]],
        obsp={
            "connectivities": adata_source.obsp["connectivities"],
            "distances": adata_source.obsp["distances"],
        },
    )

    if use_rep is not None and use_rep in adata_source.obsm:
        adata_temp.obsm[use_rep] = adata_source.obsm[use_rep]

    adata_temp.uns["neighbors"] = adata_source.uns["neighbors"].copy()
    sc.tl.umap(adata_temp, n_components=n_components, min_dist=min_dist, random_state=random_state)

    embedding = adata_temp.obsm.pop("X_umap")
    del adata_temp
    return embedding

## Deterministic Embedding Strategy

- **Stable random seed (`RANDOM_SEED`)** keeps layouts reproducible between releases, which is critical when comparing viewer builds, spotting regression diffs, or debugging quantization artifacts.
- **Stable kNN graph for all dimensions** means `sc.pp.neighbors` runs once and every 1D/2D/3D embedding encodes the exact same neighbor relationships; cross-dimensional brushing stays intuitive and centroid statistics stay comparable.
- **Shared latent representation (`latent`)** ensures the centroids and connectivities exported later line up with whatever representation was used in training; no silent drift between the viewer and the model.
- **Backed AnnData checks** let us peek into the `.h5ad` file without loading it fully and skip recomputation when the embeddings are already up to date.
- **EXPERIMENT_FILE on lustre** means the ~3.65 M-cell UMAP result is stored next to its source rather than in the project tree, keeping the repository lean.

Tweak the parameters in the previous cell only when you explicitly want to generate alternative deterministic layouts.


In [ ]:
def umap_dimensions_present(exp_file: Path, dim_keys: dict) -> tuple:
    """Return whether each required UMAP embedding is stored in exp_file plus the missing dimensions."""
    if exp_file is None or not exp_file.exists():
        return False, list(dim_keys.keys())

    backed = ad.read_h5ad(exp_file, backed="r")
    try:
        available = set(backed.obsm_keys())
    finally:
        backed.file.close()
    missing = [dim for dim, key in dim_keys.items() if key not in available]
    return len(missing) == 0, missing


def ensure_umap_embeddings():
    """Compute multi-dimensional UMAP embeddings only when they are absent on disk."""
    ready, missing_dims = umap_dimensions_present(EXPERIMENT_FILE, UMAP_DIMENSION_KEYS)

    if ready:
        print(
            f"✓ {EXPERIMENT_FILE.name} already stores "
            f"{', '.join(f'{dim}D' for dim in UMAP_DIMENSION_KEYS)} embeddings."
        )
        return

    missing_msg = ", ".join(f"{dim}D" for dim in missing_dims) if missing_dims else "all required"
    if EXPERIMENT_FILE.exists():
        print(f"Updating {EXPERIMENT_FILE.name}: missing {missing_msg} embeddings.")
    else:
        print(f"{EXPERIMENT_FILE} does not exist yet. Computing full multi-dimensional embeddings.")

    if not SOURCE_FILE.exists():
        raise FileNotFoundError(f"Source file not found: {SOURCE_FILE}")

    print(f"Loading {SOURCE_FILE.name} into memory (~3.65 M cells × 12 288 genes — this will take a while)...")
    adata = ad.read_h5ad(SOURCE_FILE)

    # Remap var.index from Ensembl gene IDs to HGNC gene symbols before saving.
    # The 'hgnc' column contains human-readable gene names (no NaN values in this dataset).
    print("Remapping var.index: Ensembl IDs → HGNC symbols via var['hgnc'] ...")
    adata.var.index = adata.var["hgnc"].values
    adata.var.index.name = "gene_name"

    print(f"Computing neighbors on obsm['{LATENT_KEY}'] (shared graph for all UMAP dimensionalities)...")
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, random_state=RANDOM_SEED, use_rep=LATENT_KEY)

    for n_dim, key in UMAP_DIMENSION_KEYS.items():
        print(f"Computing {n_dim}D UMAP → {key}")
        adata.obsm[key] = compute_umap_embedding(
            adata, n_components=n_dim, min_dist=min_dist, random_state=RANDOM_SEED
        )

    print("UMAP embeddings computed:")
    for _n_dim, key in UMAP_DIMENSION_KEYS.items():
        shape = adata.obsm[key].shape
        print(f"  {key}: {shape}")

    EXPERIMENT_FILE.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(EXPERIMENT_FILE)
    print(f"Saved updated embeddings to {EXPERIMENT_FILE}")

    del adata
    gc.collect()


ensure_umap_embeddings()

## Load UMAP run

The previous step guarantees that the experiment file exists and houses every required UMAP dimension. Load it now and double-check which embeddings are present.


In [ ]:
if not EXPERIMENT_FILE.exists():
    raise FileNotFoundError(
        f"UMAP file not found at {EXPERIMENT_FILE}. Run the cell above first."
    )

adata = ad.read_h5ad(EXPERIMENT_FILE)

# Coerce age to numeric in case it was stored as strings
if "age" in adata.obs:
    adata.obs["age"] = pd.to_numeric(adata.obs["age"], errors="raise")

# Drop internal scVI bookkeeping columns that are not meaningful to end users
drop_columns = [col for col in ("_scvi_batch", "_scvi_labels") if col in adata.obs]
if drop_columns:
    adata.obs = adata.obs.drop(columns=drop_columns)

missing_umap_keys = [key for key in UMAP_DIMENSION_KEYS.values() if key not in adata.obsm]
if missing_umap_keys:
    raise KeyError(f"Missing exact UMAP embedding keys: {missing_umap_keys}")

available_umaps = {
    f"{dim}d": adata.obsm[key] for dim, key in UMAP_DIMENSION_KEYS.items()
}
for _dim, key in UMAP_DIMENSION_KEYS.items():
    print(f"✓ Found {key}: shape {adata.obsm[key].shape}")

print(f"\nAvailable dimensions: {list(available_umaps.keys())}")
adata

## Quick UMAP stats

Lightweight sanity check on all loaded UMAP embeddings (1D, 2D, 3D).


In [ ]:
# Stats for all available UMAP dimensions
umap_stats = {}
for dim, coords in available_umaps.items():
    umap_stats[dim] = {
        "shape": coords.shape,
        "mean": coords.mean(axis=0).tolist(),
        "std": coords.std(axis=0).tolist(),
        "min": coords.min(axis=0).tolist(),
        "max": coords.max(axis=0).tolist(),
    }

print(f"UMAP stats for {adata.n_obs:,} cells:")
for dim, stats in umap_stats.items():
    print(f"\n{dim.upper()}:")
    print(f"  Shape: {stats['shape']}")
    print(f"  Mean: {[f'{x:.3f}' for x in stats['mean']]}")
    print(f"  Std:  {[f'{x:.3f}' for x in stats['std']]}")

umap_stats

## Prepare expression data

The HDCA integration file stores raw expression counts directly in `X` alongside the latent space, so no separate "complete" AnnData file is needed.  We:
- confirm gene names are already remapped to HGNC symbols (done in `ensure_umap_embeddings`),
- normalize counts to 10 000 counts per cell and log1p-transform so the quantized export stays well-behaved,
- peek at a few non-zero values as a quick sanity check.

> **Memory note:** The full 3.65 M × 12 288 expression matrix is loaded here.  On a machine with ≥ 200 GB RAM this should fit; otherwise consider chunking or exporting from backed mode.


In [ ]:
# Verify gene names on var.index are HGNC symbols (set during ensure_umap_embeddings)
print(f"var.index name : {adata.var.index.name}")
print(f"var.index (first 5): {adata.var.index[:5].tolist()}")

# Check for duplicate gene names — HGNC remapping should be unique but worth confirming
n_duplicates = adata.var.index.duplicated().sum()
if n_duplicates > 0:
    print(f"WARNING: {n_duplicates} duplicate gene names detected in var.index.")
    print("Duplicates:", adata.var.index[adata.var.index.duplicated()].tolist()[:10])
else:
    print(f"✓ All {adata.n_vars:,} gene names are unique.")

In [ ]:
# Normalize and log-transform expression counts in-place
# (adata.X is counts from the integration file — safe to transform)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata

In [ ]:
# Quick non-zero value preview on the first cell
row = adata.X[0]
dense_row = row.A.ravel() if hasattr(row, "A") else np.asarray(row).ravel()
non_zero_preview = dense_row[dense_row != 0][:5]
print(f"Non-zero log-normalized values (first cell, first 5): {non_zero_preview}")
non_zero_preview

## Export for web viewer

`cellucid.prepare` handles the heavy lifting described in `src/cellucid/prepare_data.py`:
- quantizes continuous obs/var fields and expression matrices to keep payloads small,
- auto-picks compact categorical dtypes and gzips the resulting binaries, and
- emits dataset manifests (`dataset_identity.json`, `obs_manifest.json`, `var_manifest.json`) that the WebGL viewer reads at runtime.

The call below wires our multi-dimensional UMAPs plus metadata into that exporter.  Because `var.index` is now HGNC symbols, `var_gene_id_column=None` tells the exporter to use the index directly as gene identifiers.


In [ ]:
from cellucid import prepare

In [ ]:
prepare(
    # Explicitly dimensioned UMAP embeddings
    X_umap_1d=available_umaps.get('1d'),
    X_umap_2d=available_umaps.get('2d'),
    X_umap_3d=available_umaps.get('3d'),

    # Other data matrices (scVI latent space drives centroids/kNN reuse)
    latent_space=adata.obsm[LATENT_KEY],
    obs=adata.obs,
    var=adata.var,
    gene_expression=adata.X,
    connectivities=adata.obsp['connectivities'],

    # Export behavior knobs defined inline for clarity
    var_gene_id_column=None,  # var.index already contains HGNC gene symbols
    gene_identifiers=None,    # Export every gene; slice list here if needed
    centroid_outlier_quantile=0.90,  # Trim cells far from centroid when summarizing categories
    centroid_min_points=10,          # Require at least this many cells per centroid
    force=False,
    var_quantization=8,
    obs_continuous_quantization=8,
    obs_categorical_dtype="uint16",
    compression=6,

    # Dataset identity metadata surfaced in dataset_identity.json
    out_dir=EXPORT_DIR,
    dataset_name=DATASET_NAME,
    dataset_id=DATASET_NAME,
    dataset_description="Human Developmental Cell Atlas — multi-organ single-cell transcriptomics atlas of human development",
    source_name="HDCA",
    source_url="https://www.humancellatlas.org/"
)

## Validate export artifacts

Spot-check file sizes (MB), manifest stats, and total obs/var directory sizes.


In [ ]:
import json
from pathlib import Path

BYTES_IN_MB = 1024 * 1024

def size_mb(path: Path) -> float:
    return round(path.stat().st_size / BYTES_IN_MB, 3) if path.exists() else 0

def dir_stats(path: Path) -> dict:
    if not path.exists():
        return {"size_mb": 0, "files": 0}
    total_bytes = 0
    file_count = 0
    for p in path.rglob("*"):
        if p.is_file():
            file_count += 1
            total_bytes += p.stat().st_size
    return {"size_mb": round(total_bytes / BYTES_IN_MB, 3), "files": file_count}

# Multi-dimensional point files
points_files = {
    'points_1d': EXPORT_DIR / "points_1d.bin.gz",
    'points_2d': EXPORT_DIR / "points_2d.bin.gz",
    'points_3d': EXPORT_DIR / "points_3d.bin.gz",
}

obs_manifest_path = EXPORT_DIR / "obs_manifest.json"
var_manifest_path = EXPORT_DIR / "var_manifest.json"
dataset_identity_path = EXPORT_DIR / "dataset_identity.json"
obs_dir = EXPORT_DIR / "obs"
var_dir = EXPORT_DIR / "var"

obs_manifest = json.loads(obs_manifest_path.read_text()) if obs_manifest_path.exists() else None
var_manifest = json.loads(var_manifest_path.read_text()) if var_manifest_path.exists() else None
dataset_identity = json.loads(dataset_identity_path.read_text()) if dataset_identity_path.exists() else None

# Check which point files exist
point_sizes = {}
for name, path in points_files.items():
    if path.exists():
        point_sizes[name] = size_mb(path)
        print(f"✓ {name}: {point_sizes[name]} MB")
    else:
        print(f"✗ {name}: not found")

# Show embeddings metadata
if dataset_identity and 'embeddings' in dataset_identity:
    embeddings_meta = dataset_identity['embeddings']
    print("\nEmbeddings metadata:")
    print(f"  Available dimensions: {embeddings_meta.get('available_dimensions')}")
    print(f"  Default dimension: {embeddings_meta.get('default_dimension')}D")

{
    "paths": {
        "export_dir": str(EXPORT_DIR),
        "obs_manifest": str(obs_manifest_path),
        "var_manifest": str(var_manifest_path),
        "dataset_identity": str(dataset_identity_path),
        "obs_dir": str(obs_dir),
        "var_dir": str(var_dir),
    },
    "sizes_mb": {
        **point_sizes,
        "obs_manifest": size_mb(obs_manifest_path),
        "var_manifest": size_mb(var_manifest_path),
        "dataset_identity": size_mb(dataset_identity_path),
    },
    "dir_sizes_mb": {
        "obs": dir_stats(obs_dir),
        "var": dir_stats(var_dir),
    },
    "manifest_stats": {
        "obs": None if obs_manifest is None else {
            "n_points": obs_manifest.get("n_points"),
            "fields": len(obs_manifest.get("fields", [])),
            "centroid_outlier_quantile": obs_manifest.get("centroid_outlier_quantile"),
        },
        "var": None if var_manifest is None else {
            "n_points": var_manifest.get("n_points"),
            "fields": len(var_manifest.get("fields", [])),
            "var_gene_id_column": var_manifest.get("var_gene_id_column"),
        },
        "embeddings": None if dataset_identity is None else dataset_identity.get("embeddings"),
    },
}

Done. Serve `index.html` from the repo root to view the exported data.
